# Roof + circulation

Paths people walk on, over any roof. The module takes drawn path centre lines and returns a
`Contribution`: solid to **add** (the walking surface, with ramps and stairs), space to **cut**
(headroom over it), and solid to **keep** (under the walking surface, safe from later processes).
`combine` puts it on a host. Here the host is a plain stand-in slab the size of the grid roof; later it
is the shared roof, with the other modules' contributions alongside this one.

Each step below is one group of components on a Grasshopper canvas, written as code. The logic lives in
`roof_circulation/`, where it is tested. Units are millimetres.

In [ ]:
%matplotlib inline
import roof_circulation as work

## Step 1 · The brief
*Canvas: number sliders*

The brief holds what is **controlled**: the rules every path keeps, whatever is drawn. A ramp is never
steeper than `ramp_max`; steeper runs become stairs, with no riser over `riser`. Wherever a path turns or
ends, a sloping run keeps a level landing `width / 2` long, and a flight keeps one at both ends.
`headroom` is kept empty over the path, and `support` under it is protected.

`wave` and `wavelength` set a wave along every level run and ramp. The wave obeys `ramp_max` too: it is
lowered wherever it would be steeper, so on a level run it can be at most about `wavelength / 75` high.

In [ ]:
brief = work.Brief(width=1200, ramp_max=1 / 12, riser=150, going_min=280, headroom=2100, support=300,
                   wave=80, wavelength=6000)

## Step 2 · The drawing
*Canvas: referenced polylines*

The drawing holds what is **free**: where the paths go, and the walking level at each vertex (its Z).
The sample rises from the deck to a terrace at 1550, then drops to 400, below the deck.

**Try:** give any vertex a new Z, and see where ramps turn into stairs.

In [ ]:
drawing = work.Drawing.sample()
bounds = work.steps.extent(brief, drawing)
work.preview.plan(bounds, work.steps.segments(brief, drawing), title="as drawn");

## Step 3 · The wave
*Canvas: **Divide Curve**, **Move***

`undulate` follows each level run and ramp every `wave_step` mm and lifts it by the wave. Drawn levels,
landings and stairs stay as drawn. The profile unrolls the path: distance along it against its level.

**Try:** `wave=0` for the path as drawn, or `wavelength=3000` and see the wave lowered to stay walkable.

In [ ]:
wavy = work.steps.undulate(brief, drawing)
runs = work.steps.segments(brief, wavy)
print(len(drawing.paths[0]), "drawn vertices,", len(wavy.paths[0]), "with the wave")
for seg in runs[0]:
    if seg.risers:
        print("stairs: rise", round(seg.rise), "in", seg.risers, "steps of", round(seg.going), "mm")
work.preview.profile(runs);

## Step 4 · The module, on a host
*Canvas: **Volumetric Box**, **Volumetric Boolean***

`circulation` is the whole module. `combine` adds every module's solid to the host, then cuts every
module's space, so the headroom stays clear whatever else is added. Where a path drops below the
deck, that cut opens a trench in the host.

In [ ]:
part = work.steps.circulation(brief, wavy)
host = work.steps.host(brief)
solid = work.combine(host, [part])
work.preview.section(bounds, solid, axis="y", at=2000, title="the wave, then the ramp up")
work.preview.section(bounds, solid, axis="x", at=14000, title="stairs up to the terrace")
work.preview.section(bounds, solid, axis="x", at=8000, title="stairs down into the trench");

## Step 5 · Checks
Every rule measured on the finished solid: floor underfoot and nothing overhead at every sample, not even
a 10 mm lip, and no stairs too shallow for their rise. Run this again after other modules are combined.

In [ ]:
print(work.checks.path_clear(brief, runs, solid))
print("steps too shallow:", work.checks.steep_stairs(brief, runs))

## Step 6 · Export
*Canvas: **Volumetric Mesh***

Writes the mesh (`.stl`) and the field (`.json`) to `out/`, which git ignores. Lower `voxel` for a finer mesh.

In [ ]:
vertices, faces = work.steps.export(brief, solid, bounds, "out/roof_circulation")
print("{:,} triangles".format(len(faces)))